In [ ]:
import h5py
import numpy as np
import re
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import pandas as pd
from datetime import date


In [ ]:
# ------------------------------------------------------------------
# 1. Files to scan.
#
# Each file is a Trajectum pT-study output. The two current file
# types differ in the name of their track-cut group
# (e.g. "STARTPC" vs "STARTPC200MeV"), so the group name is
# auto-detected per file below rather than hardcoded.
# ------------------------------------------------------------------
filepaths = [
    "src/monotonic_ptfluc_study/3GeV_200MeVcut_ptfluc_study.h5",
    "src/monotonic_ptfluc_study/7.7GeV_200MeVcut_ptfluc_study.h5",
    "src/monotonic_ptfluc_study/11.5GeV_200MeVcut_ptfluc_study.h5",
    "src/monotonic_ptfluc_study/19GeV_200MeVcut_ptfluc_study.h5",
    "src/monotonic_ptfluc_study/27GeV_200MeVcut_ptfluc_study.h5",
    "src/monotonic_ptfluc_study/54GeV_200MeVcut_ptfluc_study.h5",
    "src/monotonic_ptfluc_study/200GeV_200MeVcut_ptfluc_study.h5",

]


In [ ]:
# ------------------------------------------------------------------
# 2. Helper: process one file & return POINTS at the 0-5% and 30-40%
#    centrality bins.
#
# For these files, "centrality" = [2.5, 35.0] (bin centers), i.e.
# index 0 -> 0-5% and index 1 -> 30-40%. The track-cut group
# (previously hardcoded as "STARTPC" or "STARTPC200MeV" depending on
# the file) is now auto-detected, since each file only contains one
# such group. The detected group name is returned with each point so
# different file/cut types can be distinguished downstream.
# ------------------------------------------------------------------
CENT_IDX_05 = 0    # 0-5%
CENT_IDX_3040 = 1  # 30-40%


def _as_2d(a):
    a = np.asarray(a)
    return a[:, None] if a.ndim == 1 else a


def _energy_from_filename(path):
    m = re.search(r"(\d+(?:\.\d+)?)GeV", str(path))
    if not m:
        raise ValueError(f"Couldn't parse energy from filename: {path}")
    return float(m.group(1))


def _detect_track_group(hdf, base="meanptcharged"):
    """Return the single track-cut group name under `base`
    (e.g. "STARTPC" or "STARTPC200MeV"). Raises if the file
    contains more than one, in which case pass startpc_group
    explicitly to get_energy_scan_points_from_file."""
    groups = list(hdf[base].keys())
    if len(groups) != 1:
        raise ValueError(
            f"Expected exactly one track-cut group under '{base}', "
            f"found {groups}. Pass startpc_group explicitly."
        )
    return groups[0]


def get_energy_scan_points_from_file(filepath, startpc_group=None):
    """
    Returns records:
      (energy_GeV, estimator_idx, cent_label, y, yerr, track_group)

    startpc_group: name of the track-cut group to read (e.g.
    "STARTPC", "STARTPC200MeV"). If None (default), it is
    auto-detected from the file. The detected/used group name is
    returned as `track_group` on every point, so points from
    different file types (different track cuts) can be told apart
    downstream (e.g. colored separately in a plot).
    """
    E = _energy_from_filename(filepath)
    points = []

    with h5py.File(filepath, "r") as hdf:
        grp = startpc_group or _detect_track_group(hdf)

        mean_base = f"meanptcharged/{grp}/centralitybinned"
        fluc_base = f"ptfluctuationscharged/{grp}/centralitybinned"

        mean_vals = _as_2d(hdf[f"{mean_base}/values"][:])
        mean_uerr = _as_2d(hdf[f"{mean_base}/uppererrors"][:])
        mean_lerr = _as_2d(hdf[f"{mean_base}/lowererrors"][:])
        mean_sym = 0.5 * (mean_uerr + mean_lerr)

        dp_vals = _as_2d(hdf[f"{fluc_base}/values"][:])
        dp_uerr = _as_2d(hdf[f"{fluc_base}/uppererrors"][:])
        dp_lerr = _as_2d(hdf[f"{fluc_base}/lowererrors"][:])
        dp_sym = 0.5 * (dp_uerr + dp_lerr)

        # STAR-style quantity, in percent:
        with np.errstate(divide="ignore", invalid="ignore"):
            # yvals = (dp_vals / mean_vals) * 100.0
            yvals = dp_vals * 100.0
            # yerr = yvals * np.sqrt((dp_sym/dp_vals)**2 + (mean_sym/mean_vals)**2)
            yerr = yvals * np.sqrt((dp_sym / dp_vals) ** 2)

        n_cent, n_est = yvals.shape

        for cent_idx, cent_label in [(CENT_IDX_05, "0-5%"), (CENT_IDX_3040, "30-40%")]:
            for est in range(n_est):
                y = float(yvals[cent_idx, est])
                ye = float(yerr[cent_idx, est])
                if np.isfinite(y) and np.isfinite(ye):
                    points.append((E, est, cent_label, y, ye, grp))

    return points


In [ ]:
# ------------------------------------------------------------------
# 3. Collect points from all files and plot.
#
# Color encodes track-cut group / file type (e.g. STARTPC vs
# STARTPC200MeV); marker shape encodes centrality (0-5% vs 30-40%).
# A single combined legend shows both dimensions together (correct
# color + correct marker per entry).
#
# Tick/grid styling below (fixed major y-ticks at 0.1/1/2/3, log
# minor ticks, inward ticks on all sides, light grid) matches the
# STAR-style look from the original NOSMASH notebook.
# ------------------------------------------------------------------
plt.rcParams.update({
    "text.usetex": False,
    "mathtext.fontset": "cm",
    "font.family": "serif",
    "font.size": 12,
})

all_points = []
for fp in filepaths:
    all_points.extend(get_energy_scan_points_from_file(fp))

# Group by (centrality, track_group)
groups = {}
for e, est, cent, y, yerr, track_group in all_points:
    key = (cent, track_group)
    groups.setdefault(key, {"x": [], "y": [], "yerr": []})
    groups[key]["x"].append(float(e))
    groups[key]["y"].append(float(y))
    groups[key]["yerr"].append(float(yerr))


fig, ax = plt.subplots(figsize=(6, 6), dpi=200)

marker_map = {"0-5%": "s", "30-40%": "o"}

# One distinct color per track-cut group / file type
track_groups_seen = sorted({tg for (_, tg) in groups.keys()})
color_cycle = plt.rcParams["axes.prop_cycle"].by_key()["color"]
color_map = {tg: color_cycle[i % len(color_cycle)] for i, tg in enumerate(track_groups_seen)}

sorted_keys = sorted(groups.keys(), key=lambda t: (t[1], t[0]))
legend_handles = []

for cent, track_group in sorted_keys:
    d = groups[(cent, track_group)]
    x = np.array(d["x"], dtype=float)
    y = np.array(d["y"], dtype=float)
    yerr = np.array(d["yerr"], dtype=float)

    order = np.argsort(x)
    x, y, yerr = x[order], y[order], yerr[order]

    color = color_map[track_group]
    marker = marker_map.get(cent, "o")

    ax.errorbar(
        x, y, yerr=yerr,
        fmt=marker,
        color=color,
        markersize=6,
        capsize=3,
        linewidth=1.5,
    )

    legend_handles.append(
        plt.Line2D([0], [0], marker=marker, color=color, linestyle="",
                   markersize=6, label=f"{track_group} - {cent}")
    )

# Axes
ax.set_xscale("log")
ax.set_yscale("log")

ax.set_xlabel(r"Collision Energy  $\sqrt{s_{NN}}$  (GeV)")
ax.set_ylabel(r"$\sqrt{\langle \Delta p_{T,i}\Delta p_{T,j}\rangle}/\langle p_T\rangle$  [%]")

ax.set_xticks([3,7, 10, 20, 30, 50, 100, 200])
ax.set_xticklabels(["3", "7", "10", "20", "30", "50", "100", "200"])

# ax.set_ylim(1e-1, 3.5)

# ------------------------------------------------------------
# STAR-style Y ticks: label 10^-1, 1, 2, and 3
# (Major ticks fixed; minor ticks are true log subdivisions)
# ------------------------------------------------------------
ax.yaxis.set_major_locator(ticker.FixedLocator([1e-1, 1.0, 2.0, 3.0]))


class StarMajorFormatter(ticker.Formatter):
    def __call__(self, x, pos=None):
        if np.isclose(x, 1e-1):
            return r"$10^{-1}$"
        if np.isclose(x, 1.0):
            return "1"
        if np.isclose(x, 2.0):
            return "2"
        if np.isclose(x, 3.0):
            return "3"
        return ""


ax.yaxis.set_major_formatter(StarMajorFormatter())

# Minor ticks at 2..9 per decade (log-consistent)
minor_ticks = [*np.arange(1.1, 2.0, 0.1), *np.arange(2.1, 3.0, 0.1)]
ax.yaxis.set_minor_locator(ticker.FixedLocator(minor_ticks))
ax.yaxis.set_minor_formatter(ticker.NullFormatter())

# Tick styling (STAR-ish)
ax.tick_params(axis="y", which="major", length=10, width=1.2, direction="in", right=True)
ax.tick_params(axis="y", which="minor", length=5, width=1.0, direction="in", right=True)
ax.tick_params(axis="x", which="both", direction="in", top=True)

# Grid on
ax.grid(True, which="both", alpha=0.25)

ax.set_title(r"$\delta p_T / \langle p_T \rangle vs \sqrt{s_{NN}}$", fontsize=16)

# Single combined legend: color -> track cut, marker -> centrality
ax.legend(handles=legend_handles, loc="best", fancybox=True)


fig.tight_layout()
fig.savefig("graphs/pt_fluctuations_scan.png")
plt.show()

In [ ]:
# ------------------------------------------------------------------
# 4. Save the collected points to CSV.
# ------------------------------------------------------------------
df = pd.DataFrame(
    all_points,
    columns=[
        "energy_GeV",
        "estimator",
        "centrality",
        "value_percent",
        "error_percent",
        "track_group",
    ],
)

df = df.sort_values(["track_group", "centrality", "estimator", "energy_GeV"])

out_csv = f"pt_fluctuations_3-200.csv"
df.to_csv(out_csv, index=False)

print(f"Saved -> {out_csv}")
df.head()


## 5. Fit comparison (linear vs. log-linear vs. power-law vs. sqrt)

Compares candidate functional forms for $\delta p_T/\langle p_T\rangle$ vs. $\sqrt{s_{NN}}$ for each
centrality bin, using weighted least squares (weights = $1/\sigma^2$). Model selection is by
$\chi^2$/dof and AIC; a lower AIC indicates a better fit for the number of parameters used.
Also checks whether each fitted curve is monotonic over the full 3-200 GeV range.

In [ ]:
from scipy.optimize import curve_fit

def linear(E, a, b):
    return a + b * E

def loglinear(E, a, b):
    return a + b * np.log(E)

def powerlaw(E, a, b, c):
    return a + b * np.power(E, c)

def sqrtE(E, a, b):
    return a + b * np.sqrt(E)

FIT_MODELS = {
    "linear (E)":        (linear,   2, None),
    "log-linear (lnE)":  (loglinear, 2, None),
    "power law a+b*E^c": (powerlaw, 3, [1.0, 0.1, 0.5]),
    "a+b*sqrt(E)":       (sqrtE,    2, None),
}


def compare_fits(x, y, err, models=FIT_MODELS):
    """Weighted least-squares fit of several models to (x, y, err).
    Returns a list of dicts sorted by AIC (best first)."""
    n = len(x)
    results = []
    for name, (func, k, p0) in models.items():
        try:
            popt, pcov = curve_fit(
                func, x, y, sigma=err, absolute_sigma=True, p0=p0, maxfev=20000
            )
        except RuntimeError:
            continue
        yfit = func(x, *popt)
        resid = y - yfit
        chi2 = float(np.sum((resid / err) ** 2))
        dof = n - k
        aic = chi2 + 2 * k  # relative AIC (Gaussian errors, additive const drops out)

        xf = np.linspace(x.min(), x.max(), 500)
        yf = func(xf, *popt)
        dyf = np.diff(yf)
        monotonic = bool(np.all(dyf > 0) or np.all(dyf < 0))

        results.append({
            "model": name, "func": func, "popt": popt, "pcov": pcov,
            "chi2": chi2, "dof": dof,
            "chi2_per_dof": chi2 / dof if dof > 0 else np.nan,
            "aic": aic, "monotonic": monotonic,
        })
    results.sort(key=lambda r: r["aic"])
    return results


fit_results = {}
for cent in ["0-5%", "30-40%"]:
    d = df[df["centrality"] == cent].sort_values("energy_GeV")
    x = d["energy_GeV"].to_numpy()
    y = d["value_percent"].to_numpy()
    err = d["error_percent"].to_numpy()
    fit_results[cent] = compare_fits(x, y, err)

    print(f"\n{cent}")
    print(f"{'model':<20}{'chi2':>8}{'dof':>5}{'chi2/dof':>10}{'AIC':>8}  monotonic")
    for r in fit_results[cent]:
        print(f"{r['model']:<20}{r['chi2']:>8.3f}{r['dof']:>5}"
              f"{r['chi2_per_dof']:>10.3f}{r['aic']:>8.3f}  {str(r['monotonic']):>9}")


## 6. Monotonicity significance

For each centrality bin: (a) probability that each adjacent step is an increase, using the
combined error of the two neighboring points, and (b) the joint probability of a fully
monotonically-increasing sequence across all 7 energies, via Monte Carlo sampling of each
point from its own Gaussian uncertainty.

In [ ]:
from scipy import stats

rng = np.random.default_rng(0)
N_MC = 1_000_000

for cent in ["0-5%", "30-40%"]:
    d = df[df["centrality"] == cent].sort_values("energy_GeV")
    x = d["energy_GeV"].to_numpy()
    y = d["value_percent"].to_numpy()
    err = d["error_percent"].to_numpy()

    print(f"\n=== {cent} ===")
    for i in range(len(x) - 1):
        diff = y[i + 1] - y[i]
        combined_err = np.sqrt(err[i] ** 2 + err[i + 1] ** 2)
        z = diff / combined_err
        p_increase = stats.norm.cdf(z)
        print(f"  E={x[i]:>6} -> E={x[i+1]:>6}:  P(increase) = {p_increase:.3f}  (z={z:+.2f})")

    samples = rng.normal(loc=y, scale=err, size=(N_MC, len(y)))
    is_monotonic = np.all(np.diff(samples, axis=1) > 0, axis=1)
    p_joint = is_monotonic.mean()
    tau, p_tau = stats.kendalltau(x, y)
    print(f"  Joint P(fully monotonic, all 7 pts) = {p_joint:.3f}")
    print(f"  Kendall\'s tau = {tau:.3f}  (p={p_tau:.3f})")


## 7. Best-fit plot (best model per centrality, overlaid on data)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6), dpi=200)

colors = {"0-5%": color_cycle[0], "30-40%": color_cycle[1]}
markers = {"0-5%": "s", "30-40%": "o"}

xf_full = np.linspace(3, 200, 500)

for cent in ["0-5%", "30-40%"]:
    d = df[df["centrality"] == cent].sort_values("energy_GeV")
    x = d["energy_GeV"].to_numpy()
    y = d["value_percent"].to_numpy()
    err = d["error_percent"].to_numpy()

    best = fit_results[cent][0]
    yf = best["func"](xf_full, *best["popt"])

    ax.errorbar(
        x, y, yerr=err, fmt=markers[cent], color=colors[cent],
        markersize=6, capsize=3, linewidth=1.5,
        label=f"{cent} data",
    )
    ax.plot(
        xf_full, yf, "--", color=colors[cent], linewidth=1.5,
        label=f"{cent} best fit: {best['model']} ($\\chi^2$/dof={best['chi2_per_dof']:.2f})",
    )

ax.set_xscale("log")
ax.set_xlabel(r"Collision Energy  $\sqrt{s_{NN}}$  (GeV)")
ax.set_ylabel(r"$\sqrt{\langle \Delta p_{T,i}\Delta p_{T,j}\rangle}/\langle p_T\rangle$  [%]")
ax.set_xticks([3, 7, 10, 20, 30, 50, 100, 200])
ax.set_xticklabels(["3", "7", "10", "20", "30", "50", "100", "200"])
ax.grid(True, which="both", alpha=0.25)
ax.set_title(r"$\delta p_T/\langle p_T\rangle$ vs $\sqrt{s_{NN}}$: best fit per centrality", fontsize=14)
ax.legend(loc="best", fontsize=9)

fig.tight_layout()
fig.savefig("graphs/pt_fluctuations_bestfit.png")
plt.show()
